# Official CODI KV spectral causality on Kaggle

Run the frozen-checkpoint rank-four retain/remove experiment on a Kaggle T4 or P100. The notebook evaluates learned student KV directions against energy-matched random directions on full GSM8K. Position 4 and position 5 run first and form the primary hypothesis family.

Before starting, enable a GPU and Internet. Attach a Kaggle input dataset containing the completed `official_codi_kv_subspaces/n5000_seed1/statistics.pt` file from Drive.

## 1. Configure the run

Use Save Version and Run All after the smoke test succeeds. Replace `RUN_COMMIT` with the immutable commit printed by setup. `RESUME_INPUT` is only needed when continuing from a previously saved Kaggle output dataset.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Pin the printed commit before Save Version and Run All.
REPO_DIR = "/kaggle/working/latent-reasoning"

# Leave empty to auto-discover an attached n5000_seed1/statistics.pt.
STATISTICS_INPUT = ""
# Optional attached previous Kaggle output or exported dataset.
RESUME_INPUT = ""

RUN_SMOKE = True
RUN_FULL_EVALUATION = True
RUN_ANALYSIS = True
UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-kv-causal"

RANK = 4
RIDGE_RATIO = 1e-4
RANDOM_SEED = 20260727
POSITIONS = [0, 1, 2, 3, 4, 5]
PRIMARY_POSITIONS = [4, 5]
BATCH_SIZE = 128
MAX_NEW_TOKENS = 256
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0
FAMILYWISE_ALPHA = 0.05
PRECISION = "float16"

## 2. Clone the pinned code and install dependencies

In [ ]:
import datetime
import hashlib
import json
import os
import pathlib
import shutil
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL RUN:", commit)

## 3. Verify the Kaggle GPU and implementation

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)

## 4. Locate the completed calibration statistics

The file is read-only. The exporter verifies its schema, completed count, checkpoint identity, embedded official-reproduction gate, finite moments, and minimum 5,000-example contract.

In [ ]:
if STATISTICS_INPUT:
    STATISTICS = pathlib.Path(STATISTICS_INPUT)
else:
    candidates = [
        path
        for path in pathlib.Path("/kaggle/input").rglob("statistics.pt")
        if "n5000_seed1" in path.as_posix()
    ]
    assert len(candidates) == 1, (
        "Expected one attached n5000_seed1/statistics.pt, found "
        f"{len(candidates)}. Set STATISTICS_INPUT explicitly. Candidates: {candidates}"
    )
    STATISTICS = candidates[0]
assert STATISTICS.is_file(), f"Missing calibration statistics: {STATISTICS}"
print("Calibration statistics:", STATISTICS)
print("Size:", STATISTICS.stat().st_size / 2**30, "GiB")

## 5. Restore a previous partial Kaggle run

Skip this on the first run. When resuming, attach the prior saved output as a Kaggle input and set `RESUME_INPUT` to that dataset root. Completed condition summaries and JSONL files are copied back and verified by the evaluator.

In [ ]:
WORK_OUTPUT_ROOT = repo / "outputs" / "official_codi_kv_causal"
WORK_REPORT_ROOT = repo / "reports" / "official_codi_kv_causal"
WORK_LOG_ROOT = repo / "logs" / "official_codi_kv_causal"
for path in (WORK_OUTPUT_ROOT, WORK_REPORT_ROOT, WORK_LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

if RESUME_INPUT:
    resume_root = pathlib.Path(RESUME_INPUT)
    manifests = list(resume_root.rglob("official_codi_kv_causal/full_gsm8k/run_manifest.json"))
    assert len(manifests) == 1, f"Expected one causal run manifest, found {manifests}"
    source_output = manifests[0].parents[1]
    shutil.copytree(source_output, WORK_OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored:", source_output)
    restored = json.loads((WORK_OUTPUT_ROOT / "full_gsm8k" / "run_manifest.json").read_text())
    print("Previously completed conditions:", len(restored.get("completed_conditions", [])))
else:
    print("Starting without a previous causal evaluation")

## 6. Export compact learned and energy-matched random bases

In [ ]:
SUBSPACE_ARTIFACT = WORK_OUTPUT_ROOT / "subspaces" / "student_rank4.pt"
SUBSPACE_MANIFEST = SUBSPACE_ARTIFACT.with_suffix(".json")
SUBSPACE_ARTIFACT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        sys.executable, "scripts/export_official_codi_student_subspaces.py",
        "--statistics", str(STATISTICS),
        "--output", str(SUBSPACE_ARTIFACT),
        "--rank", str(RANK),
        "--ridge-ratio", str(RIDGE_RATIO),
        "--random-seed", str(RANDOM_SEED),
        "--minimum-examples", "5000",
    ],
    cwd=REPO_DIR,
    check=True,
)
subspace_manifest = json.loads(SUBSPACE_MANIFEST.read_text())
assert subspace_manifest["state"] == "complete"
assert subspace_manifest["processed_examples"] == 5000
for kind in ("key", "value"):
    diagnostic = subspace_manifest["diagnostics"][kind]
    assert diagnostic["energy_match_max_relative_error"] < 1e-5
    assert diagnostic["learned_orthonormal_max_error"] < 1e-4
    assert diagnostic["random_orthonormal_max_error"] < 1e-4
print(json.dumps(subspace_manifest, indent=2))

## 7. Persistent local runner

Kaggle saves `/kaggle/working` when the notebook version finishes. The causal evaluator also writes each condition atomically, so a saved partial output can be attached and resumed in a later version.

In [ ]:
def run_persisted(command, log_name):
    log_path = WORK_LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    print("Persistent session log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
        log.flush()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}; inspect {log_path}")
    return log_path

## 8. Run a 32-example smoke test

The completed statistics embed the previously passed official reproduction gate. This cell verifies the real checkpoint, cache hook, and position-4/5 intervention paths before the long run.

In [ ]:
SMOKE_ROOT = WORK_OUTPUT_ROOT / "smoke_gsm8k"
if RUN_SMOKE:
    run_persisted(
        [
            sys.executable, "-u", "scripts/run_official_codi_kv_causal.py",
            "--config", "configs/official_codi_gpt2.yaml",
            "--subspace-artifact", str(SUBSPACE_ARTIFACT),
            "--output-dir", str(SMOKE_ROOT),
            "--positions", "4,5",
            "--rank", str(RANK),
            "--limit", "32",
            "--batch-size", "32",
            "--max-new-tokens", str(MAX_NEW_TOKENS),
            "--seed", str(RANDOM_SEED),
            "--precision", PRECISION,
            "--device", "cuda",
        ],
        "smoke_gsm8k.log",
    )
    smoke = json.loads((SMOKE_ROOT / "run_manifest.json").read_text())
    assert smoke["state"] == "complete"
    assert smoke["evaluated_count"] == 32
    assert len(smoke["conditions"]) == 9
    print(json.dumps(json.loads((SMOKE_ROOT / "summary.json").read_text()), indent=2))
else:
    print("Smoke test skipped")

## 9. Run all 29 full-GSM8K conditions

The unchanged baseline runs first and must reproduce the official accuracy gate. Position 4 and position 5 interventions run next. Positions 0 through 3 and the all-position interventions follow as secondary diagnostics.

In [ ]:
FULL_ROOT = WORK_OUTPUT_ROOT / "full_gsm8k"
if RUN_FULL_EVALUATION:
    run_persisted(
        [
            sys.executable, "-u", "scripts/run_official_codi_kv_causal.py",
            "--config", "configs/official_codi_gpt2.yaml",
            "--subspace-artifact", str(SUBSPACE_ARTIFACT),
            "--output-dir", str(FULL_ROOT),
            "--positions", ",".join(map(str, POSITIONS)),
            "--include-all",
            "--rank", str(RANK),
            "--limit", "0",
            "--batch-size", str(BATCH_SIZE),
            "--max-new-tokens", str(MAX_NEW_TOKENS),
            "--seed", str(RANDOM_SEED),
            "--precision", PRECISION,
            "--device", "cuda",
        ],
        "full_gsm8k.log",
    )
    full_manifest = json.loads((FULL_ROOT / "run_manifest.json").read_text())
    assert full_manifest["state"] == "complete"
    assert full_manifest["evaluated_count"] == 1319
    assert len(full_manifest["conditions"]) == 29
    assert set(full_manifest["completed_conditions"]) == set(full_manifest["conditions"])
    print("All 29 full-GSM8K conditions are complete")
else:
    print("Full evaluation skipped")

## 10. Run the paired causal analysis

In [ ]:
from IPython.display import Markdown, display

REPORT_PATH = WORK_REPORT_ROOT / "official_codi_rank4_full_gsm8k.json"
if RUN_ANALYSIS:
    subprocess.run(
        [
            sys.executable, "scripts/analyze_official_codi_kv_causal.py",
            "--evaluation-root", str(FULL_ROOT),
            "--output", str(REPORT_PATH),
            "--primary-positions", ",".join(map(str, PRIMARY_POSITIONS)),
            "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
            "--seed", str(BOOTSTRAP_SEED),
            "--familywise-alpha", str(FAMILYWISE_ALPHA),
        ],
        cwd=REPO_DIR,
        check=True,
    )
    report = json.loads(REPORT_PATH.read_text())
    assert report["evaluated_count"] == 1319
    display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
else:
    print("Analysis skipped")

## 11. Build a durable Kaggle output package

The package excludes the multi-gigabyte calibration input and Hugging Face cache. It contains the compact bases, every evaluation arm, the report, logs, commit identity, and checksums.

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_kv_causal_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(WORK_OUTPUT_ROOT, export_repo / "outputs" / "official_codi_kv_causal")
shutil.copytree(WORK_REPORT_ROOT, export_repo / "reports" / "official_codi_kv_causal")
shutil.copytree(WORK_LOG_ROOT, export_repo / "logs" / "official_codi_kv_causal")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")

files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
checksum_lines = []
for path in files:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksum_lines.append(f"{digest}  {path.relative_to(EXPORT_ROOT).as_posix()}")
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(checksum_lines) + "\n")

print("Export root:", EXPORT_ROOT)
print("Files:", len(files) + 1)
print("Size:", sum(path.stat().st_size for path in EXPORT_ROOT.rglob("*") if path.is_file()) / 2**20, "MiB")
print("Use Save Version with outputs enabled. The browser does not need to remain open during a committed Kaggle run.")

## 12. Optional direct dataset upload

Save Version is sufficient. Enable this only after the complete report exists and your Kaggle credentials can create or update the configured dataset.

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(
        KAGGLE_DATASET_HANDLE,
        str(EXPORT_ROOT),
        version_notes=f"Official CODI rank-4 KV causal evaluation at {commit}",
    )
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Direct dataset upload skipped. Save this notebook version with outputs enabled.")